In [ ]:
import os
from tqdm import tqdm
import json

from transformers import AutoTokenizer
from datasets import load_dataset

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("openai/gpt-oss-120b")

## nvidia/Nemotron-SFT-SWE-v2

In [ ]:
# hf download nvidia/Nemotron-SFT-SWE-v2 --max-workers 64 --repo-type dataset
data_path = "/mnt/md0/hf_cache/datasets--nvidia--Nemotron-SFT-SWE-v2/snapshots/bd151f3f2d89c4804dda0083d912bd9f6a0a9fb7/data/"

agentless_samples = []
with open(os.path.join(data_path, "agentless.jsonl"), "r") as f:
    for line in tqdm(f):
        agentless_samples.append(json.loads(line.strip()))

# swe_samples = []
# with open(os.path.join(data_path, "swe.jsonl"), "r") as f:
#     for line in tqdm(f):
#         swe_samples.append(json.loads(line.strip()))

In [ ]:
# Using only agentless, since SWE doesn't have the reasoning content and we aren't doing SDG

In [ ]:
def clean_messages(messages, strip_null_tools: bool = False):
    for message in messages:
        if "reasoning_content" in message:
            message["thinking"] = message.pop("reasoning_content")
        if message.get("content") is None:
            message["content"] = ""
        if message.get("thinking") is None:
            message["thinking"] = ""
        if strip_null_tools and len(message.get("tool_calls", []) or []) == 0:
            message.pop("tool_calls", None)
        if strip_null_tools and message.get("function_call") is None:
            message.pop("function_call", None)
    # Remove any turns marked "assistant" that have no content, thinking, or tool calls, since these are likely artifacts of the data collection process and don't provide useful signal for pretraining.
    # Mutate the messages object in place to ensure changes propagate to the sample dict that contains it.
    messages[:] = [m for m in messages if not (m["role"] == "assistant" and m["content"] == "" and m["thinking"] == "" and len(m.get("tool_calls", [])) == 0)]
    return messages

def clean_tools(tools):
    """Recursively remove None values from tool schemas (e.g. default: null)."""
    if isinstance(tools, dict):
        return {k: clean_tools(v) for k, v in tools.items() if v is not None}
    if isinstance(tools, list):
        return [clean_tools(t) for t in tools]
    return tools


In [ ]:
swe_agentless_all = open("sft-swe-v2-agentless.jsonl", "w")

agentless_samples = agentless_samples[0:20000]

# Preprocessing pass: efficient
chat_templated_samples = []
for sample in tqdm(agentless_samples):
    messages = clean_messages(sample["messages"], strip_null_tools=True)
    tools = clean_tools(sample.get("tools", []))

    templated = tokenizer.apply_chat_template(messages, tokenize=False, tools=tools, enable_thinking=True)
    chat_templated_samples.append(templated)

# Write out samples with token counts: efficient
for sample in tqdm(agentless_samples, total=len(agentless_samples)):
    dump_sample = json.dumps(sample)
    swe_agentless_all.write(dump_sample + "\n")

swe_agentless_all.close()

## nvidia/Nemotron-SFT-Agentic-v2

In [ ]:
# hf download nvidia/Nemotron-SFT-Agentic-v2 --max-workers 64 --repo-type dataset
data_path = "/mnt/md0/hf_cache/datasets--nvidia--Nemotron-SFT-Agentic-v2/snapshots/49e79a3be5ab8cf7511a12958b95cfd6408cd8db/data/"

interactive_agent_samples = []
with open(os.path.join(data_path, "interactive_agent.jsonl"), "r") as f:
    for line in tqdm(f):
        if len(interactive_agent_samples) > 50000:
            break
        try:
            interactive_agent_samples.append(json.loads(line.strip()))
        except:
            continue

search_samples = []
with open(os.path.join(data_path, "search.jsonl"), "r") as f:
    for line in tqdm(f):
        try:
            search_samples.append(json.loads(line.strip()))
        except:
            continue

tool_calling_samples = []
with open(os.path.join(data_path, "tool_calling.jsonl"), "r") as f:
    for line in tqdm(f):
        try:
            tool_calling_samples.append(json.loads(line.strip()))
        except:
            continue

In [ ]:
len(interactive_agent_samples), len(search_samples), len(tool_calling_samples)

In [ ]:
# Both Interactive Agent and Search samples are fairly short seqlen and have reasoning content
# Search samples are super long and probably better suited for RL phase

In [ ]:
interactive_all = open("sft-agentic-v2-interactive.jsonl", "w")

# Preprocessing pass: efficient
chat_templated_samples = []
for sample in tqdm(interactive_agent_samples):
    messages = clean_messages(sample["messages"])
    tools = clean_tools(sample.get("tools", []))

    templated = tokenizer.apply_chat_template(messages, tokenize=False, tools=tools, enable_thinking=True)
    chat_templated_samples.append(templated)

# Write out samples with token counts: efficient
num_written = 0
for sample in tqdm(interactive_agent_samples, total=len(interactive_agent_samples)):
    if sample["metadata"]["turn_token_count"][-1]["reasoning"] <= 0:
        continue
    dump_sample = json.dumps(sample)
    interactive_all.write(dump_sample + "\n")
    num_written += 1
    if num_written > 15_000:
        break

interactive_all.close()

In [ ]:
search_all = open("sft-agentic-v2-search.jsonl", "w")

# Preprocessing pass: efficient
chat_templated_samples = []
for sample in tqdm(search_samples):
    messages = clean_messages(sample["messages"], strip_null_tools=True)
    tools = clean_tools(sample.get("tools", []))

    try:
        templated = tokenizer.apply_chat_template(messages, tokenize=False, tools=tools, enable_thinking=True)
        chat_templated_samples.append(templated)
    except Exception:
        chat_templated_samples.append(None)

# Write out samples with token counts: efficient
for sample, templated in tqdm(zip(search_samples, chat_templated_samples), total=len(search_samples)):
    if templated is None:
        continue
    if sample["metadata"]["turn_token_count"][-1]["reasoning"] <= 0:
        continue
    dump_sample = json.dumps(sample)
    search_all.write(dump_sample + "\n")

search_all.close()

In [ ]:
tool_calling_all = open("sft-agentic-v2-tool-calling.jsonl", "w")

# Preprocessing pass: efficient
chat_templated_samples = []
for sample in tqdm(tool_calling_samples):
    messages = clean_messages(sample["messages"])
    tools = clean_tools(sample.get("tools", []))

    templated = tokenizer.apply_chat_template(messages, tokenize=False, tools=tools, enable_thinking=True)
    chat_templated_samples.append(templated)

# Write out samples with token counts: efficient
for sample in tqdm(tool_calling_samples, total=len(tool_calling_samples)):
    if sample["metadata"]["turn_token_count"][-1]["reasoning"] <= 0:
        continue
    dump_sample = json.dumps(sample)
    tool_calling_all.write(dump_sample + "\n")

tool_calling_all.close()

In [ ]:
num_tokens_per_sample = []
for batch_idx in tqdm(range(0, len(chat_templated_samples), 1000)):
    batch = chat_templated_samples[batch_idx:batch_idx+1000]
    batch_to_tokenize = [b for b in batch if b is not None]
    tokenized = tokenizer(batch_to_tokenize, truncation=False)
    num_skipped = 0 # Handle skipping inline to save another full pass over the entire dataset
    for i in range(len(batch)):
        if batch[i] is None:
            num_tokens_per_sample.append(0)
            num_skipped += 1
        else:
            num_tokens = len(tokenized["input_ids"][i - num_skipped])
            num_tokens_per_sample.append(num_tokens)
    break

## nvidia/Nemotron-Math-v2

In [ ]:
# GPT-OSS-120B traces for low, medium, and high reasoning effort.
# Some problems have multiple entries in each split, so we use a set to remove the duplicates at each reasoning level.
# Duplicates across levels are okay for now, and may be filtered out in later stages.

# hf download nvidia/Nemotron-Math-v2 --max-workers 64 --repo-type dataset
data_path = "/mnt/md0/hf_cache/datasets--nvidia--Nemotron-Math-v2/snapshots/8e793210e175b6406c752a870f585f62de98c0d3/data"

high_problem_ids = set()
high_samples = []
for fname in ("high.part_00.jsonl", "high.part_01.jsonl", "high.part_02.jsonl"):
    with open(os.path.join(data_path, fname), "r") as f:
        for line in tqdm(f):
            if len(high_samples) > 30000:
                break
            try:
                sample = json.loads(line.strip())
                if sample["problem"] not in high_problem_ids:
                    high_samples.append(sample)
                    high_problem_ids.add(sample["problem"])
            except:
                continue

medium_problem_ids = set()
medium_samples = []
with open(os.path.join(data_path, "high.jsonl"), "r") as f:
    for line in tqdm(f):
        if len(high_samples) > 30000:
            break
        try:
            sample = json.loads(line.strip())
            if sample["problem"] not in high_problem_ids:
                medium_samples.append(sample)
                medium_problem_ids.add(sample["problem"])
        except:
            continue

low_problem_ids = set()
low_samples = []
with open(os.path.join(data_path, "low.jsonl"), "r") as f:
    for line in tqdm(f):
        if len(low_samples) > 30000:
            break
        try:
            sample = json.loads(line.strip())
            if sample["problem"] not in low_problem_ids:
                low_samples.append(sample)
                low_problem_ids.add(sample["problem"])
        except:
            continue

In [ ]:
# An acceptable number of samples for each reasoning level, after deduplication.
len(high_samples), len(medium_samples), len(low_samples)

In [ ]:
reasoning_high_all = open("math-v2-high.jsonl", "w")

# Preprocessing pass: efficient
chat_templated_samples = []
for sample in tqdm(high_samples):
    messages = clean_messages(sample["messages"], strip_null_tools=True)
    tools = clean_tools(sample.get("tools", []))

    try:
        templated = tokenizer.apply_chat_template(messages, tokenize=False, tools=tools, enable_thinking=True, reasoning_effort="high")
    except Exception:
        # Some very rare cases of broken tool calls in this dataset (1 in a million), so we skip those.
        chat_templated_samples.append(None)
        continue
    chat_templated_samples.append(templated)

# Write out samples with token counts: efficient
num_written = 0
for sample in tqdm(high_samples, total=len(high_samples)):
    if num_tokens == 0:
        continue
    if "metadata" in sample and "turn_token_count" in sample["metadata"] and sample["metadata"]["turn_token_count"][-1]["reasoning"] <= 0:
        continue
    sample["reasoning_effort"] = "high"
    dump_sample = json.dumps(sample)
    reasoning_high_all.write(dump_sample + "\n")
    num_written += 1
    if num_written > 15_000:
        break

reasoning_high_all.close()

In [ ]:
reasoning_medium_all = open("math-v2-medium.jsonl", "w")

# Preprocessing pass: efficient
chat_templated_samples = []
for sample in tqdm(medium_samples):
    messages = clean_messages(sample["messages"], strip_null_tools=True)
    tools = clean_tools(sample.get("tools", []))

    try:
        templated = tokenizer.apply_chat_template(messages, tokenize=False, tools=tools, enable_thinking=True, reasoning_effort="medium")
    except Exception:
        # Some very rare cases of broken tool calls in this dataset (1 in a million), so we skip those.
        chat_templated_samples.append(None)
        continue
    chat_templated_samples.append(templated)

# Write out samples with token counts: efficient
num_written = 0
for sample in tqdm(medium_samples, total=len(medium_samples)):
    if num_tokens == 0:
        continue
    if "metadata" in sample and "turn_token_count" in sample["metadata"] and sample["metadata"]["turn_token_count"][-1]["reasoning"] <= 0:
        continue
    sample["reasoning_effort"] = "medium"
    dump_sample = json.dumps(sample)
    reasoning_medium_all.write(dump_sample + "\n")
    num_written += 1
    if num_written > 15_000:
        break

reasoning_medium_all.close()

In [ ]:
reasoning_low_all = open("math-v2-low.jsonl", "w")

# Preprocessing pass: efficient
chat_templated_samples = []
for sample in tqdm(low_samples):
    messages = clean_messages(sample["messages"], strip_null_tools=True)
    tools = clean_tools(sample.get("tools", []))

    try:
        templated = tokenizer.apply_chat_template(messages, tokenize=False, tools=tools, enable_thinking=True, reasoning_effort="medium")
    except Exception:
        # Some very rare cases of broken tool calls in this dataset (1 in a million), so we skip those.
        chat_templated_samples.append(None)
        continue
    chat_templated_samples.append(templated)

# Write out samples with token counts: efficient
num_written = 0
for sample in tqdm(low_samples, total=len(low_samples)):
    if "metadata" in sample and "turn_token_count" in sample["metadata"] and sample["metadata"]["turn_token_count"][-1]["reasoning"] <= 0:
        continue
    sample["reasoning_effort"] = "low"
    dump_sample = json.dumps(sample)
    reasoning_low_all.write(dump_sample + "\n")
    num_written += 1
    if num_written > 5_000:
        break

reasoning_low_all.close()

## nvidia/Nemotron-Competitive-Programming-v1

In [ ]:
# Has three splits: competitive coding CPP, competitive coding Python, and Infinibyte.
# All of them may end up exceeding context for pretrain run, so we'll load only the first 1000 samples from each to check.

# hf download nvidia/Nemotron-Competitive-Programming-v1 --max-workers 64 --repo-type dataset
data_path = "/mnt/md0/hf_cache/datasets--nvidia--Nemotron-Competitive-Programming-v1/snapshots/d6e7c6b404ed5db6e1104b41d0f80a0c7dad7bf8/data/"

cpp_samples = []
for fname in ("competitive_coding_cpp.part_00.jsonl", "competitive_coding_cpp.part_01.jsonl"):
    with open(os.path.join(data_path, fname), "r") as f:
        for line in tqdm(f):
            try:
                sample = json.loads(line.strip())
                cpp_samples.append(sample)
                if len(cpp_samples) >= 50000:
                    break
            except:
                continue

python_samples = []
for fname in ("competitive_coding_python.part_00.jsonl", "competitive_coding_python.part_01.jsonl"):
    with open(os.path.join(data_path, fname), "r") as f:
        for line in tqdm(f):
            try:
                sample = json.loads(line.strip())
                python_samples.append(sample)
                if len(python_samples) >= 50000:
                    break
            except:
                continue

infinibyte_samples = []
for fname in ("infinibyte.part_00.jsonl", "infinibyte.part_01.jsonl"):
    with open(os.path.join(data_path, fname), "r") as f:
        for line in tqdm(f):
            try:
                sample = json.loads(line.strip())
                infinibyte_samples.append(sample)
                if len(infinibyte_samples) >= 50000:
                    break
            except:
                continue

In [ ]:
# Patch Missing Samples
from tqdm import tqdm
from datasets import load_dataset

# NOTE: Need to download the datasets and remove the scripts (TACO.py, apps.py)
hf_datasets = {
    "taco": load_dataset("/mnt/md0/hf_cache/datasets--BAAI--TACO/snapshots/d593ed0a2becbbc952230bb89be09189bf1056dc/", trust_remote_code=True),
    "apps": load_dataset("/mnt/md0/hf_cache/datasets--codeparrot--apps/snapshots/21e74ddf8de1a21436da12e3e653065c5213e9d1", trust_remote_code=True),
    "code_contests": load_dataset("deepmind/code_contests"),
    "open-r1/codeforces": load_dataset("open-r1/codeforces")
}

def get_question(ds_name, split, index):
    benchmark = hf_datasets[ds_name][split][int(index)]
    if ds_name == "code_contests":
        if not benchmark["description"]:
            print("No description codecontests")
            return None
        return benchmark["description"]
    elif ds_name in ["taco", "apps"]:
        return benchmark["question"]
    elif ds_name == "open-r1/codeforces":
        if not benchmark["description"]:
            print("No description codeforces")
            return None
        question = benchmark["description"]
        if benchmark["input_format"]:
            question += "\n\nInput\n\n" + benchmark["input_format"]
        if benchmark["output_format"]:
            question += "\n\nOutput\n\n" + benchmark["output_format"]
        if benchmark["examples"]:
            question += "\n\nExamples"
            for example in benchmark["examples"]:
                if "input" in example:
                    question += "\n\nInput\n\n" + example["input"]
                if "output" in example:
                    question += "\n\nOutput\n\n" + example["output"]
        if benchmark["note"]:
            question += "\n\nNote\n\n" + benchmark["note"]
        return question

    print("No match")

    return None

num_skipped_cpp = 0
num_replaced_cpp = 0
for sample in cpp_samples:
    assert sample["dataset"] in ["taco", "apps", "code_contests", "open-r1/codeforces"]
    ds_name, ds_split, ds_index = sample["dataset"], sample["split"], int(sample["index"])
    question = get_question(ds_name, ds_split, ds_index)
    assert question is not None
    got_match = False
    num_skipped_cpp += 1
    for m in sample["messages"]:
        if m["role"] == "user" and m["content"] == "-":
            assert question != "-"
            m["content"] = question
            num_replaced_cpp += 1
            num_skipped_cpp -= 1
            break
print(f"{num_skipped_cpp=} {num_replaced_cpp=}")

num_skipped_py = 0
num_replaced_py = 0

for sample in python_samples:
    assert sample["dataset"] in ["taco", "apps", "code_contests", "open-r1/codeforces"]
    ds_name, ds_split, ds_index = sample["dataset"], sample["split"], int(sample["index"])
    question = get_question(ds_name, ds_split, ds_index)
    assert question is not None
    got_match = False
    num_skipped_py += 1
    for m in sample["messages"]:
        if m["role"] == "user" and m["content"] == "-":
            assert question != "-"
            m["content"] = question
            num_replaced_py += 1
            num_skipped_py -= 1
            break
print(f"{num_skipped_py=} {num_replaced_py=}")

In [ ]:
sample

In [ ]:
python_samples[10000]

In [ ]:
cpp_all = open("competitive-coding-v1-cpp.jsonl", "w")

# Preprocessing pass: efficient
keep_samples = []
chat_templated_samples = []
for sample in tqdm(cpp_samples):
    messages = clean_messages(sample["messages"], strip_null_tools=True)
    tools = clean_tools(sample.get("tools", []))

    templated = tokenizer.apply_chat_template(messages, tokenize=False, tools=tools, enable_thinking=True)
    keep_samples.append(sample)
    chat_templated_samples.append(templated)

# Write out samples with token counts: efficient
num_written = 0
for sample in tqdm(keep_samples, total=len(keep_samples)):
    if "metadata" in sample and "turn_token_count" in sample["metadata"] and sample["metadata"]["turn_token_count"][-1]["reasoning"] <= 0:
        continue
    dump_sample = json.dumps(sample)
    cpp_all.write(dump_sample + "\n")
    num_written += 1
    if num_written >= 25_000:
        break

cpp_all.close()

In [ ]:
python_all = open("competitive-coding-v1-python.jsonl", "w")

# Preprocessing pass: efficient
keep_samples = []
chat_templated_samples = []
for sample in tqdm(python_samples):
    messages = clean_messages(sample["messages"], strip_null_tools=True)
    tools = clean_tools(sample.get("tools", []))

    templated = tokenizer.apply_chat_template(messages, tokenize=False, tools=tools, enable_thinking=True)
    keep_samples.append(sample)
    chat_templated_samples.append(templated)

# Write out samples with token counts: efficient
num_written = 0
for sample in tqdm(keep_samples, total=len(keep_samples)):
    if "metadata" in sample and "turn_token_count" in sample["metadata"] and sample["metadata"]["turn_token_count"][-1]["reasoning"] <= 0:
        continue
    dump_sample = json.dumps(sample)
    python_all.write(dump_sample + "\n")
    num_written += 1
    if num_written >= 35_000:
        break

python_all.close()

In [ ]:
infinibyte_all = open("competitive-coding-v1-infinibyte.jsonl", "w")

# Preprocessing pass: efficient
keep_samples = []
chat_templated_samples = []
for sample in tqdm(infinibyte_samples):
    messages = clean_messages(sample["messages"], strip_null_tools=True)
    tools = clean_tools(sample.get("tools", []))

    templated = tokenizer.apply_chat_template(messages, tokenize=False, tools=tools, enable_thinking=True)
    keep_samples.append(sample)
    chat_templated_samples.append(templated)

# Write out samples with token counts: efficient
num_written = 0
for sample in tqdm(keep_samples, total=len(keep_samples)):
    if "metadata" in sample and "turn_token_count" in sample["metadata"] and sample["metadata"]["turn_token_count"][-1]["reasoning"] <= 0:
        continue
    dump_sample = json.dumps(sample)
    infinibyte_all.write(dump_sample + "\n")
    num_written += 1
    if num_written >= 10_000:
        break

infinibyte_all.close()

## nvidia/Nemotron-SFT-Instruction-Following-Chat-v2

In [ ]:
# Use the reasoning-on subset
data_path = "/mnt/md0/hf_cache/datasets--nvidia--Nemotron-SFT-Instruction-Following-Chat-v2/snapshots/1a9454ed054b8544503ab8d8c0a519d141a44c5b/data/"

chat_samples = []
with open(os.path.join(data_path, "reasoning_on.jsonl"), "r") as f:
    for line in tqdm(f):
        chat_samples.append(json.loads(line.strip()))

In [ ]:
chat_all = open("sft-if-chat.jsonl", "w")

# Preprocessing pass: efficient
keep_samples = []
chat_templated_samples = []
for sample in tqdm(chat_samples):
    messages = clean_messages(sample["messages"], strip_null_tools=True)
    tools = clean_tools(sample.get("tools", []))

    try:
        templated = tokenizer.apply_chat_template(messages, tokenize=False, tools=tools, enable_thinking=True)
    except Exception:
        # Some completions seem to contain copies of the reserved characters. Skip them.
        continue
    keep_samples.append(sample)
    chat_templated_samples.append(templated)
    if len(keep_samples) > 30000:
        break

# Write out samples with token counts: efficient
num_written = 0
for sample in tqdm(keep_samples, total=len(keep_samples)):
    if "metadata" in sample and "turn_token_count" in sample["metadata"] and sample["metadata"]["turn_token_count"][-1]["reasoning"] <= 0:
        continue
    dump_sample = json.dumps(sample)
    chat_all.write(dump_sample + "\n")
    num_written += 1
    if num_written >= 15_000:
        break

chat_all.close()

## nvidia/Nemotron-Instruction-Following-Chat-v1

In [ ]:
# Use the reasoning-on subset
data_path = "/mnt/md0/hf_cache/datasets--nvidia--Nemotron-Instruction-Following-Chat-v1/snapshots/83dcd3aded0d289b0bbc018d3f9af4c5dd4005df/data/"

structured_output_samples = []
with open(os.path.join(data_path, "structured_outputs.jsonl"), "r") as f:
    for line in tqdm(f):
        structured_output_samples.append(json.loads(line.strip()))

In [ ]:
structured_output_all = open("if-chat-v1-structured_outputs.jsonl", "w")

# Preprocessing pass: efficient
keep_samples = []
chat_templated_samples = []
for sample in tqdm(structured_output_samples):
    messages = clean_messages(sample["messages"], strip_null_tools=True)
    tools = clean_tools(sample.get("tools", []))

    try:
        templated = tokenizer.apply_chat_template(messages, tokenize=False, tools=tools, enable_thinking=True)
    except Exception:
        # Some completions seem to contain copies of the reserved characters. Skip them.
        continue
    keep_samples.append(sample)
    chat_templated_samples.append(templated)

# Write out samples with token counts: efficient
for sample in tqdm(keep_samples, total=len(keep_samples)):
    if "metadata" in sample and "turn_token_count" in sample["metadata"] and sample["metadata"]["turn_token_count"][-1]["reasoning"] <= 0:
        continue
    dump_sample = json.dumps(sample)
    structured_output_all.write(dump_sample + "\n")

structured_output_all.close()

## nvidia/Nemotron-SFT-Multilingual-v1

In [ ]:
# Has about 18 splits for multilingual tasks. We'll put all of them into one list for easy processing.
# Downsample by 4x when loading data to keep it manageable, since there are about 3M samples in total.

# hf download nvidia/Nemotron-SFT-Multilingual-v1 --max-workers 64 --repo-type dataset
data_path = "/mnt/md0/hf_cache/datasets--nvidia--Nemotron-SFT-Multilingual-v1/snapshots/22c86505762a7c595abee309d720084351c9f4ba/data/"
# glob all files in data_path
multilingual_samples = []
num_samples_processed = 0
for fname in os.listdir(data_path):
    if fname.endswith(".jsonl"):
        with open(os.path.join(data_path, fname), "r") as f:
            for line in tqdm(f):
                try:
                    multilingual_samples.append(json.loads(line.strip()))
                    num_samples_processed += 1
                    if num_samples_processed >= 25000:
                        break
                except:
                    continue

In [ ]:
multilingual_all = open("sft-multilingual-v1.jsonl", "w")

# Preprocessing pass: efficient
keep_samples = []
chat_templated_samples = []
for sample in tqdm(multilingual_samples):
    messages = clean_messages(sample["messages"], strip_null_tools=True)
    tools = clean_tools(sample.get("tools", []))

    try:
        templated = tokenizer.apply_chat_template(messages, tokenize=False, tools=tools, enable_thinking=True)
    except Exception:
        # Some completions seem to contain copies of the reserved characters. Skip them.
        continue
    keep_samples.append(sample)
    chat_templated_samples.append(templated)

# Write out samples with token counts: efficient
num_written = 0
for sample in tqdm(keep_samples, total=len(keep_samples)):
    if "metadata" in sample and "turn_token_count" in sample["metadata"] and sample["metadata"]["turn_token_count"][-1]["reasoning"] <= 0:
        continue
    dump_sample = json.dumps(sample)
    multilingual_all.write(dump_sample + "\n")
    num_written += 1
    if num_written >= 10000:
        break

multilingual_all.close()

## nvidia/Nemotron-Science-v1

In [ ]:
# hf download nvidia/Nemotron-Science-v1 --max-workers 64 --repo-type dataset
data_path = "/mnt/md0/hf_cache/datasets--nvidia--Nemotron-Science-v1/snapshots/82e1af468197076b4f0f392c239274eac032adc7/data/"
science_samples = []
for fname in os.listdir(data_path):
    if fname.endswith(".jsonl"):
        with open(os.path.join(data_path, fname), "r") as f:
            for line in tqdm(f):
                try:
                    science_samples.append(json.loads(line.strip()))
                except:
                    continue

In [ ]:
science = open("science-v1.jsonl", "w")


# Preprocessing pass: efficient
keep_samples = []
chat_templated_samples = []
for sample in tqdm(science_samples):
    messages = clean_messages(sample["messages"], strip_null_tools=True)
    tools = clean_tools(sample.get("tools", []))

    try:
        templated = tokenizer.apply_chat_template(messages, tokenize=False, tools=tools, enable_thinking=True)
    except Exception:
        # Some completions seem to contain copies of the reserved characters. Skip them.
        continue
    keep_samples.append(sample)
    chat_templated_samples.append(templated)

# Batch tokenization: efficient
num_tokens_per_sample = []
for batch_idx in tqdm(range(0, len(chat_templated_samples), 1000)):
    batch = chat_templated_samples[batch_idx:batch_idx+1000]
    tokenized = tokenizer(batch, truncation=False)
    num_skipped = 0 # Handle skipping inline to save another full pass over the entire dataset
    for i in range(len(batch)):
        num_tokens = len(tokenized["input_ids"][i - num_skipped])
        num_tokens_per_sample.append(num_tokens)

# Write out samples with token counts: efficient
N=0
for sample, num_tokens in tqdm(zip(keep_samples, num_tokens_per_sample), total=len(keep_samples)):
    if num_tokens == 0:
        continue
    if "metadata" in sample and "turn_token_count" in sample["metadata"] and sample["metadata"]["turn_token_count"][-1]["reasoning"] <= 0:
        continue
    dump_sample = json.dumps(sample)
    science.write(dump_sample + "\n")
    N+=1
    if N>10000:
        break

science.close()

# Collation

In [ ]:
training_data_files_dir = "/root/eagle/ModelOptNew/eagle_training_data/long_context/"
training_data_files = os.listdir(training_data_files_dir)
training_data_files = [f for f in training_data_files if f.endswith(".jsonl")]
categories = [f.replace(".jsonl", "").replace('-4k', '') for f in training_data_files]

In [ ]:
all_samples = []
for idx, f in enumerate(training_data_files):
    with open(os.path.join(training_data_files_dir, f), "r") as f:
        for line in tqdm(f):
            try:
                sample = json.loads(line.strip())
                sample["source_dataset"] = categories[idx]
                all_samples.append(sample)
            except:
                continue

In [ ]:
all_samples = [s for s in all_samples if s["messages"][-1]["role"] == "assistant" and s["messages"][-1].get("thinking", "")]

num_invalid = {}
invalid_idxs = []
for idx, sample in enumerate(tqdm(all_samples)):
    messages = sample["messages"]
    tools = clean_tools(sample.get("tools", []))

    try:
        kwargs = {}
        if "reasoning_effort" in sample:
            kwargs["reasoning_effort"] = sample["reasoning_effort"]
        templated = tokenizer.apply_chat_template(messages, tokenize=False, tools=tools, enable_thinking=True, **kwargs)
        sample["templated"] = templated
    except Exception:
        num_invalid[sample["source_dataset"]] = num_invalid.get(sample["source_dataset"], 0) + 1
        invalid_idxs.append(idx)
        continue

In [ ]:
num_invalid

In [ ]:
# Trim down the samples to a fixed set of columns:
# - messages
# - source_dataset
# - reasoning_effort (default to "medium")
# - tools (default to empty list)
# - templated (the result of applying the chat template, for reference)
# - metadata (default to empty dict)
formatted_all_samples = []
for idx in range(len(all_samples)):
    if idx in invalid_idxs:
        continue
    sample = all_samples[idx]
    formatted_sample = {
        "messages": sample["messages"],
        "source_dataset": sample["source_dataset"],
        "reasoning_effort": sample.get("reasoning_effort", "medium"),
        "tools": clean_tools(sample.get("tools", [])),
        "templated": sample["templated"],
        "metadata": sample.get("metadata", {}),
    }
    formatted_all_samples.append(formatted_sample)

In [ ]:
with open("all_longcontext_samples_with_templates.jsonl", "w") as f:
    for sample in formatted_all_samples:
        dump_sample = json.dumps(sample)
        f.write(dump_sample + "\n")

In [ ]:
# Count the lines in each file and print it out
lines_per_category = {}
for sample in formatted_all_samples:
    category = sample["source_dataset"]
    lines_per_category[category] = lines_per_category.get(category, 0) + 1

In [ ]:
lines_per_category

In [ ]:
import matplotlib.pyplot as plt

data = lines_per_category

labels = list(data.keys())
sizes = list(data.values())
labels[3], labels[8] = labels[8], labels[3]
sizes[3], sizes[8] = sizes[8], sizes[3]

total = sum(sizes)
def format_absolute_k(pct):
    # Reverse-engineer the absolute value from the percentage
    absolute_val = (pct / 100.0) * total
    
    # Round to the nearest thousand and format as an integer with 'k'
    rounded_k = int(round(absolute_val / 1000))
    return f"{rounded_k}k"

plt.figure(figsize=(9, 6))
plt.pie(
    sizes, 
    labels=labels,
    # autopct='%1.1f%%',
    autopct=format_absolute_k,
    startangle=160,
    pctdistance=0.8
)

plt.title("GPT-OSS EAGLE3 Long Context Dataset Mix")

plt.axis('equal')
plt.tight_layout()

# Save the plot
plt.savefig("gpt-oss-eagle3-longcontext-dataset-mix.png", dpi=300)

plt.show()
